# Sample 01: ជជែករហ័ស តាមរយៈ OpenAI SDK

កំណត់ត្រានេះបង្ហាញពីរបៀបប្រើ OpenAI SDK ជាមួយ Microsoft Foundry Local សម្រាប់ការជជែករហ័ស។

## សង្ខេប

ឧទាហរណ៍នេះបង្ហាញ៖
- ការប្រើប្រាស់ OpenAI Python SDK ជាមួយ Foundry Local
- ការដោះស្រាយ និងគ្រប់គ្រងទាំង Azure OpenAI និង ការកំណត់រចនាសម្ព័ន្ធ Foundry ក្នុងស្រុក
- អនុវត្តការគ្រប់គ្រងកំហុសយ៉ាងត្រឹមត្រូវ និងយុទ្ធសាស្ត្រជំនួស
- ការប្រើ FoundryLocalManager សម្រាប់ការគ្រប់គ្រងសេវាកម្ម


## តម្រូវការ​មុន

សូមប្រាកដថាអ្នកមាន៖
1. Foundry Local បានដំឡើង និងកំពុងដំណើរការ
2. ម៉ូឌែលបានផ្ទុក (ឧ. `foundry model run phi-4-mini`)
3. បានដំឡើងកញ្ចប់ Python ដែលត្រូវការ

### ដំឡើងការពឹងផ្អែក


In [ ]:
# Install required packages if not already installed
!pip install openai foundry-local-sdk

## នាំចូលបណ្ណាល័យដែលចាំបាច់


In [ ]:
import os
import sys
from openai import OpenAI

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, will use manual configuration")

## ការកំណត់

កំណត់ការកំណត់របស់អ្នក។ អ្នកអាចប្រើឯណាមួយពីរនេះ៖
1. **Azure OpenAI** (ផ្អែកលើពពក)
2. **Foundry Local** (សន្និដ្ឋានក្នុងស្រុក)

### ជម្រើស 1: ការកំណត់ Azure OpenAI


In [ ]:
# Azure OpenAI Configuration (uncomment and set your values)
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"

### ជម្រើស 2: ការកំណត់ក្នុងស្រុករបស់ Foundry


In [ ]:
# Foundry Local Configuration
MODEL_ALIAS = "phi-4-mini"  # Change this to your preferred model
BASE_URL = "http://localhost:8000"  # Default Foundry Local URL
API_KEY = ""  # Usually not needed for local

## ការចាប់ផ្តើមអតិថិជន

បង្កើតអតិថិជន OpenAI ដោយផ្អែកលើការកំណត់រចនាសម្ព័ន្ធរបស់អ្នក:


In [ ]:
def create_client():
    """Create and configure OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if azure_endpoint and azure_api_key:
        # Azure OpenAI path
        model = os.environ.get("MODEL", "your-deployment-name")
        client = OpenAI(
            base_url=f"{azure_endpoint}/openai",
            api_key=azure_api_key,
            default_query={"api-version": azure_api_version},
        )
        print(f"🌐 Using Azure OpenAI with model: {model}")
        return client, model
    
    # Foundry Local path
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            manager = FoundryLocalManager(MODEL_ALIAS)
            model_info = manager.get_model_info(MODEL_ALIAS)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            model = model_info.id
            print(f"🏠 Using Foundry Local SDK with model: {model}")
            return client, model
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{BASE_URL}/v1",
        api_key=API_KEY
    )
    model = MODEL_ALIAS
    print(f"🔧 Using manual configuration with model: {model}")
    return client, model

# Initialize the client
client, model = create_client()

## ឧទាហរណ៍ជជែកមូលដ្ឋាន

យើងសាកល្បងអន្តរកម្មជជែកសាមញ្ញមួយ:


In [ ]:
def chat_with_model(prompt, max_tokens=128):
    """Send a chat message to the model and get a response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test with a simple prompt
prompt = "Say hello from Foundry Local and explain what you are in one sentence."
print(f"👤 User: {prompt}")
print("\n🤖 Assistant:")
response = chat_with_model(prompt)
print(response)

## សម័យជជែកអន្តរកម្ម

សាកល្បងប្រភេទផ្សេងៗនៃសំណើ (prompts) ដើម្បីមើលថា ម៉ូឌែលឆ្លើយតបយ៉ាងដូចម្តេច:


In [ ]:
# Example prompts to try
example_prompts = [
    "What are the benefits of edge AI?",
    "Explain the difference between local and cloud AI inference.",
    "Write a short Python function to calculate the factorial of a number.",
    "What is Microsoft Foundry Local?"
]

for i, prompt in enumerate(example_prompts, 1):
    print(f"\n{'='*50}")
    print(f"Example {i}: {prompt}")
    print(f"{'='*50}")
    
    response = chat_with_model(prompt, max_tokens=200)
    print(response)

## ការប្រើប្រាស់កម្រិតខ្ពស់: ការឆ្លើយតបជាស្ទ្រីម

សម្រាប់ចម្លើយដែលវែងជាងនេះ ការឆ្លើយតបជាស្ទ្រីមអាចផ្តល់បទពិសោធន៍ប្រើប្រាស់ដែលកាន់តែប្រសើរ:


In [ ]:
def chat_with_streaming(prompt, max_tokens=300):
    """Send a chat message with streaming response."""
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        print("🤖 Assistant (streaming):")
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        return f"Error: {e}"

# Test streaming with a longer prompt
streaming_prompt = "Explain the advantages of running AI models locally on device versus in the cloud. Include privacy, latency, and cost considerations."
print(f"👤 User: {streaming_prompt}\n")
streaming_response = chat_with_streaming(streaming_prompt)

## ការដោះស្រាយកំហុស និង ការធ្វើវាយតម្លៃ

យើងមកពិនិត្យស្ថានភាពសេវាកម្ម និងម៉ូឌែលដែលអាចប្រើបាន:


In [ ]:
def check_service_health():
    """Check Foundry Local service health."""
    try:
        # Try to list available models
        models_response = client.models.list()
        models = [model.id for model in models_response.data]
        
        print("✅ Service is healthy")
        print(f"📋 Available models: {models}")
        print(f"🎯 Current model: {model}")
        
        if model in models:
            print("✅ Current model is available")
        else:
            print("⚠️ Current model may not be loaded")
            
    except Exception as e:
        print(f"❌ Service check failed: {e}")
        print("\n🔧 Troubleshooting tips:")
        print("1. Make sure Foundry Local is running")
        print("2. Check if a model is loaded: foundry service ps")
        print("3. Verify the endpoint URL is correct")

check_service_health()

## ការធ្វើតេស្តសំណើផ្ទាល់ខ្លួន

ប្រើកោសិកាខាងក្រោមដើម្បីសាកល្បងសំណើផ្ទាល់ខ្លួនរបស់អ្នក:


In [ ]:
# Enter your custom prompt here
custom_prompt = "Write a haiku about artificial intelligence running on edge devices."

print(f"👤 User: {custom_prompt}\n")
custom_response = chat_with_model(custom_prompt, max_tokens=100)
print(f"🤖 Assistant: {custom_response}")

## សេចក្តីសង្ខេប

កំណត់ត្រានេះ បង្ហាញពី៖

1. **✅ ការកំណត់អតិថិជន**: របៀបកំណត់ OpenAI SDK ជាមួយ Foundry Local
2. **✅ ជជែកមូលដ្ឋាន**: ប្រតិកម្មសំណើ-ចម្លើយសាមញ្ញ
3. **✅ ស្ទ្រីម**: ការបង្កើតចម្លើយក្នុងពេលពិត
4. **✅ ការគ្រប់គ្រងកំហុស**: ការគ្រប់គ្រងកំហុស និងការធ្វើវិភាគកំហុសយ៉ាងរឹងមាំ
5. **✅ សុខភាពសេវា**: ពិនិត្យភាពអាចប្រើបានរបស់ម៉ូឌែល និងស្ថានភាពសេវា

### ជំហានបន្ទាប់

- **Sample 02**: ការរួមបញ្ចូល SDK កម្រិតខ្ពស់ ជាមួយការគាំទ្រ Azure OpenAI
- **Sample 04**: ការបង្កើតកម្មវិធីជជែក Chainlit
- **Sample 05**: ប្រព័ន្ធរៀបចំសម្រាប់ភ្នាក់ងារច្រើន
- **Sample 06**: ការបញ្ជូនម៉ូឌែលដោយឆ្លាត

### អត្ថប្រយោជន៍សំខាន់ៗនៃ Foundry Local

- 🔒 **ឯកជនភាព**: ទិន្នន័យមិនដែលចេញពីឧបករណ៍របស់អ្នក
- ⚡ **ល្បឿន**: ការទាយលទ្ធផលក្នុងស្រុកដោយពេលយឺតទាប
- 💰 **ថ្លៃ**: គ្មានចំណាយសម្រាប់ការប្រើប្រាស់ API
- 🔌 **ក្រៅបណ្ដាញ**: ដំណើរការបានដោយគ្មានការតភ្ជាប់អ៊ីនធឺណិត
- 🛠️ **ភាពឆបគ្នា**: API ដែលផ្គូផ្គងជាមួយ OpenAI


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការដកខ្លួនពីការទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ទោះបីយើងខិតខំរកភាពត្រឹមត្រូវ ក៏សូមមេត្តាជ្រាបថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមិនត្រឹមត្រូវខ្លះៗ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកថាជាធនធាន​ដែលមានអាទិភាព។ សម្រាប់ព័ត៌មានសំខាន់ សូមពិចារណាបកប្រែដោយអ្នកជំនាញមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
